# <font color="#418FDE" size="6.5" uppercase>**Foundations of Bias and Fairness in AI**</font>

>Last update: 20250116.
    
By the end of this Lecture, you will be able to:
- Identify common sources of bias in AI systems.
- Distinguish between bias and fairness in AI.
- Evaluate various fairness metrics and their trade-offs.


## **1. Understanding Human Bias in AI Development**

### **1.1. Influence of Human Judgment in Data Annotation**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_01_01.png" width="250">



>* Human judgment introduces bias in data annotation.
* Bias affects AI performance across demographics.

>* Lack of standardized guidelines causes inconsistencies.
* Comprehensive training needed to reduce biases.

>* Commercial pressures increase biased annotations.
* Implement quality control to reduce bias.



### **1.2. Tracing the Roots of Systematic Biases**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_01_02.png" width="250">



>* Biases stem from societal structures and inequities.
* Underrepresentation leads to skewed AI outcomes.

>* Algorithms reflect flawed assumptions and simplifications.
* Feedback loops amplify existing algorithmic biases.

>* Lack of diversity leads to biased outcomes.
* Diverse teams create fairer AI systems.



### **1.3. Navigating Cultural Nuances in AI Deployment**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_01_03.png" width="250">



>* Cultural biases affect AI system fairness.
* Underrepresented groups face AI discrimination risks.

>* Cultural nuances affect AI user interaction.
* Privacy norms influence AI system engagement.

>* Engage communities to understand cultural needs.
* Incorporate diverse datasets and local stakeholders.



### **1.4. Impact of Training Data Bias on AI Predictions**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_01_04.png" width="250">



>* Training data bias skews AI predictions.
* Facial recognition errors highlight racial biases.

>* NLP models learn biases from internet text.
* Gender bias affects AI decision-making processes.

>* Diversify datasets and use synthetic data.
* Monitor AI systems for emerging biases.



In [ ]:
#@title Python Code - Impact of Training Data Bias on AI Predictions

# This code demonstrates the impact of training data bias using the Titanic dataset.
# We create a biased training dataset by under-representing male passengers
# to observe the effects of bias on model performance.

import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Load the Titanic dataset from seaborn
data = sns.load_dataset('titanic')

# Drop rows with missing values
data.dropna(subset=['survived', 'sex', 'age', 'pclass', 'fare'], inplace=True)

# Encode categorical variables
label_encoders = {}
for column in ['sex', 'embarked', 'class', 'who', 'alone']:
    le = LabelEncoder()
    data[column] = le.fit_transform(data[column])
    label_encoders[column] = le

# Select features and target
X = data[['sex', 'age', 'pclass', 'fare']]
y = data['survived']

# Standardize numerical features
scaler = StandardScaler()
X[['age', 'fare']] = scaler.fit_transform(X[['age', 'fare']])

# Check the distribution of the target variable
print("Original target distribution:")
print(y.value_counts())

# Introduce bias: Under-represent male passengers
male_passengers = data[data['sex'] == 1].sample(frac=0.2, random_state=42)  # Keep 20% of males
female_passengers = data[data['sex'] == 0]
biased_data = pd.concat([male_passengers, female_passengers])

# Create features and target for the biased dataset
X_biased = biased_data[['sex', 'age', 'pclass', 'fare']]
y_biased = biased_data['survived']

# Split original and biased datasets into training and test sets
X_train_original, X_test_original, y_train_original, y_test_original = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_biased, X_test_biased, y_train_biased, y_test_biased = train_test_split(X_biased, y_biased, test_size=0.2, random_state=42)

# Train a Logistic Regression model on the biased dataset
model_biased = LogisticRegression(max_iter=1000)
model_biased.fit(X_train_biased, y_train_biased)

# Evaluate the biased model on the original test set
y_pred_biased = model_biased.predict(X_test_original)
print("\nPerformance on the original test set with the biased model:")
print(classification_report(y_test_original, y_pred_biased))
print(confusion_matrix(y_test_original, y_pred_biased))

# Train a Logistic Regression model on the original dataset
model_original = LogisticRegression(max_iter=1000)
model_original.fit(X_train_original, y_train_original)

# Evaluate the original model on the original test set
y_pred_original = model_original.predict(X_test_original)
print("\nPerformance on the original test set with the original model:")
print(classification_report(y_test_original, y_pred_original))
print(confusion_matrix(y_test_original, y_pred_original))


### **1.5. Continuous Data Evaluation and Adjustment**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_01_05.png" width="250">



>* Regularly audit datasets for biases and imbalances.
* Adjust training data for equitable representation.

>* Integrate feedback loops for real-time bias detection.
* Enhance fairness and accuracy through user feedback.

>* Ethical guidelines ensure transparency and accountability.
* Diverse teams help identify overlooked biases.



In [ ]:
#@title Python Code - Continuous Data Evaluation and Adjustment

# This code demonstrates how to perform continuous data evaluation and adjustment using a Random Forest classifier on the adult dataset.
# The goal is to evaluate the model's performance and adjust the training data based on the results to mitigate bias.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Step 1: Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]
data = pd.read_csv(url, names=columns, sep=",\s*", engine="python")

# Step 2: Preprocess the dataset
data = data.replace("?", np.nan).dropna()
data = data[(data['race'] == 'White') | (data['race'] == 'Black')]  # Focus on race bias
data['sex'] = data['sex'].apply(lambda x: 1 if x == 'Male' else 0)  # Encode sex
data['race'] = data['race'].apply(lambda x: 1 if x == 'White' else 0)  # Encode race
data['income'] = data['income'].apply(lambda x: 1 if x == '>50K' else 0)  # Encode target

# Select features and target
features = ['age', 'education_num', 'hours_per_week', 'sex', 'race']
X = data[features]
y = data['income']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Train the initial model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
print("Initial Model Performance:")
print(classification_report(y_test, y_pred))

# Step 4: Bias Evaluation
# Check model predictions based on 'race' and 'sex'
def evaluate_bias(group_name, group_value):
    group_data = X_test[X_test[group_name] == group_value]
    group_labels = y_test[group_data.index]
    group_preds = model.predict(group_data)
    return accuracy_score(group_labels, group_preds)

bias_results = {
    "White": evaluate_bias("race", 1),
    "Black": evaluate_bias("race", 0),
    "Male": evaluate_bias("sex", 1),
    "Female": evaluate_bias("sex", 0)
}
print("Bias Evaluation Results:")
print(bias_results)

# Step 5: Bias Mitigation
# Adjust training data to balance representation
white_minority = data[(data['race'] == 1) & (data['income'] == 1)]
black_minority = data[(data['race'] == 0) & (data['income'] == 1)]

adjusted_data = pd.concat([white_minority, black_minority, data[data['income'] == 0]])
X_adjusted = adjusted_data[features]
y_adjusted = adjusted_data['income']

X_train_adj, X_test_adj, y_train_adj, y_test_adj = train_test_split(
    X_adjusted, y_adjusted, test_size=0.2, random_state=42
)

# Retrain the model
model.fit(X_train_adj, y_train_adj)

# Evaluate the adjusted model
y_pred_adj = model.predict(X_test_adj)
print("Adjusted Model Performance:")
print(classification_report(y_test_adj, y_pred_adj))

# Re-evaluate bias
adjusted_bias_results = {
    "White": evaluate_bias("race", 1),
    "Black": evaluate_bias("race", 0),
    "Male": evaluate_bias("sex", 1),
    "Female": evaluate_bias("sex", 0)
}
print("Adjusted Bias Evaluation Results:")
print(adjusted_bias_results)


## **2. Clarifying Bias and Fairness in AI**

### **2.1. Understanding Explicit and Implicit Bias in AI**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_02_01.png" width="250">



>* Explicit bias is intentional and identifiable.
* Implicit bias is unconscious and harder to detect.

>* Explicit bias mitigated by diverse datasets.
* Implicit bias addressed with nuanced techniques.

>* AI systems must be socially responsible.
* Addressing biases ensures fairer AI systems.



In [ ]:
#@title Python Code - Understanding Explicit and Implicit Bias in AI

# This code demonstrates how to evaluate explicit and implicit bias in a machine learning model using the Adult Income dataset.
# The dataset contains demographic information and aims to predict whether an individual's income exceeds $50K/year.
# We will train a Random Forest classifier and analyze the model's predictions for bias based on gender and race.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load the Adult Income dataset from UCI Machine Learning Repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
column_names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=column_names, na_values=' ?', skipinitialspace=True)

# Drop rows with missing values
df.dropna(inplace=True)

# Convert categorical variables to numerical using one-hot encoding
df = pd.get_dummies(df, drop_first=True)

# Define features and target variable
X = df.drop('income_>50K', axis=1)
Y = df['income_>50K']

# Split the dataset into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Initialize and train the Random Forest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, Y_train)

# Make predictions on the test set
Y_pred = model.predict(X_test)

# Evaluate the model's performance
print("Confusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))
print("Classification Report:")
print(classification_report(Y_test, Y_pred))

# Analyze bias in predictions based on gender and race
# Calculate the proportion of positive predictions for each gender
gender_bias = df.groupby('sex_Male')['income_>50K'].mean()
print("Gender Bias Analysis:")
print(gender_bias)

# Calculate the proportion of positive predictions for each race
race_bias = df.groupby('race_White')['income_>50K'].mean()
print("Race Bias Analysis:")
print(race_bias)


### **2.2. Balancing Group and Individual Fairness in AI**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_02_02.png" width="250">



>* Group fairness ensures similar outcomes for groups.
* Individual fairness treats similar individuals similarly.

>* Balancing group and individual fairness is challenging.
* Fairness constraints help manage trade-offs effectively.

>* AI in law enforcement balances fairness needs.
* Fairness-aware algorithms require transparency and community input.



In [ ]:
#@title Python Code - Balancing Group and Individual Fairness in AI

# The code demonstrates how to evaluate and address fairness in machine learning models by balancing group fairness (e.g., demographic parity across genders) and individual fairness (consistent treatment of similar individuals).
# It highlights the trade-offs between accuracy and fairness by applying fairness constraints, which reduce group disparities but slightly decrease overall accuracy.
# Additionally, it showcases potential individual fairness issues, emphasizing the need for comprehensive fairness strategies in AI systems.

# Install the fairlearn library (uncomment the line below if not already installed)
!pip install fairlearn

# Import the necessary libraries for data processing, modeling, and fairness evaluation
import pandas as pd  # Library for data manipulation
import numpy as np  # Library for numerical computations
from sklearn.model_selection import train_test_split  # Function to split data into train and test sets
from sklearn.ensemble import RandomForestClassifier  # Random Forest classifier for modeling
from sklearn.metrics import accuracy_score  # Metric to calculate model accuracy
from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference  # Fairness metrics
from fairlearn.postprocessing import ThresholdOptimizer  # Post-processing tool for fairness adjustments
from sklearn.preprocessing import LabelEncoder  # For encoding categorical data into numerical format

# Define the URL for the dataset and the column names
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]

# Load the dataset from the URL and replace missing values (denoted as '?') with NaN
data = pd.read_csv(data_url, names=columns, sep=',\s+', na_values=["?"])

# Drop rows with missing values
data.dropna(inplace=True)

# Encode the 'income' column: assign 1 to '>50K' and 0 to '<=50K'
data['income'] = data['income'].apply(lambda x: 1 if x == '>50K' else 0)

# Encode the 'sex' column: assign 0 to Female and 1 to Male
data['sex'] = LabelEncoder().fit_transform(data['sex'])

# Perform one-hot encoding on categorical columns and drop the first category to avoid multicollinearity
data = pd.get_dummies(data, columns=['workclass', 'education', 'marital-status',
                                     'occupation', 'relationship', 'race', 'native-country'], drop_first=True)

# Separate the dataset into features (X) and target labels (y)
X = data.drop('income', axis=1)
y = data['income']

# Split the data into training and testing sets (70% training, 30% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize a Random Forest Classifier with a fixed random state for reproducibility
model = RandomForestClassifier(random_state=42)

# Train the model using the training data
model.fit(X_train, y_train)

# Use the trained model to make predictions on the test data
y_pred = model.predict(X_test)

# Calculate the accuracy of the model on the test data
accuracy = accuracy_score(y_test, y_pred)

# Print the overall accuracy of the model
print(f"Accuracy: {accuracy:.2f}")

# Define the sensitive feature ('sex') for fairness evaluation
sensitive_feature = X_test['sex']

# Create a MetricFrame to evaluate accuracy across different groups based on the sensitive feature
metric_frame = MetricFrame(
    metrics={'accuracy': accuracy_score},  # Metric to calculate for each group
    y_true=y_test,  # True labels of the test set
    y_pred=y_pred,  # Predicted labels from the model
    sensitive_features=sensitive_feature  # Sensitive feature to group by
)

# Print the accuracy for each group based on the sensitive feature
print("Group fairness metrics:")
print(metric_frame.by_group)

# Initialize the ThresholdOptimizer for fairness-aware post-processing
threshold_optimizer = ThresholdOptimizer(
    estimator=model,  # Base machine learning model
    constraints="demographic_parity",  # Fairness constraint to satisfy
    predict_method="auto"  # Automatically adjust predictions
)

# Fit the ThresholdOptimizer using the training data and the sensitive feature
threshold_optimizer.fit(X_train, y_train, sensitive_features=X_train['sex'])

# Generate fairness-adjusted predictions on the test data
y_pred_fair = threshold_optimizer.predict(X_test, sensitive_features=sensitive_feature)

# Calculate the accuracy of the fairness-adjusted predictions
fair_accuracy = accuracy_score(y_test, y_pred_fair)

# Calculate the demographic parity difference of the fairness-adjusted predictions
demographic_parity = demographic_parity_difference(y_test, y_pred_fair, sensitive_features=sensitive_feature)

# Print the accuracy and demographic parity difference for the fairness-adjusted model
print(f"Fair Accuracy: {fair_accuracy:.2f}")
print(f"Demographic Parity Difference: {demographic_parity:.2f}")

# Define a function to test for individual fairness by introducing slight variations to input samples
def individual_fairness_check(data, model, num_samples=100):
    # Randomly select a sample of indices from the dataset
    sample_indices = np.random.choice(data.index, num_samples, replace=False)
    for idx in sample_indices:
        # Extract the original sample and create a copy with a slight modification
        sample = data.loc[idx:idx]
        perturbed_sample = sample.copy()
        perturbed_sample['age'] = sample['age'] + 1  # Modify the 'age' feature slightly

        # Predict the labels for the original and perturbed samples
        pred_original = model.predict(sample.drop('income', axis=1))
        pred_perturbed = model.predict(perturbed_sample.drop('income', axis=1))

        # Check if predictions for the original and perturbed samples differ
        if pred_original != pred_perturbed:
            print(f"Potential individual unfairness detected for sample {idx}.")

# Run the individual fairness check on the dataset
individual_fairness_check(data, model)


### **2.3. Ethical Theories and Fairness in AI**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_02_03.png" width="250">



>* Utilitarianism maximizes happiness, may neglect minorities.
* Deontological ethics ensures dignity, prevents discrimination.

>* Virtue ethics focuses on moral character.
* AI should embody fairness, honesty, empathy.

>* Justice as fairness aligns with AI principles.
* Design AI to benefit the least advantaged.



### **2.4. Equitable Distribution Strategies in AI Systems**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_02_04.png" width="250">



>* Ensure AI benefits all societal segments.
* Diversify datasets for robust, fair AI.

>* AI impacts job markets, needs equitable strategies.
* Implement fairness-aware algorithms and retraining programs.

>* AI optimizes public resource distribution.
* Fairness constraints ensure equitable resource allocation.



### **2.5. Navigating Societal Norms and Legal Frameworks in AI Fairness**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_02_05.png" width="250">



>* Cultural norms shape AI fairness perceptions.
* Legal frameworks vary, impacting global AI deployment.

>* Legal frameworks demand AI decision transparency.
* Cultural norms affect AI transparency perception.

>* AI fairness evolves with societal values.
* Legal updates address AI bias challenges.



## **3. Exploring Fairness Metrics and Ethical Considerations**

### **3.1. Comparing Statistical Parity and Equal Opportunity**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_03_01.png" width="250">



>* Statistical parity ensures equal outcomes across groups.
* May lead to reverse discrimination and merit issues.

>* Ensures equal chances for similarly qualified individuals.
* Challenges in measuring qualifications and controlling factors.

>* Trade-offs between fairness in outcomes and processes.
* Fairness metric selection should be context-dependent.



In [ ]:
#@title Python Code - Comparing Statistical Parity and Equal Opportunity

# his code demonstrates how to evaluate fairness metrics like statistical parity and equal opportunity by training a logistic regression model on the Adult Income dataset.
# It calculates these metrics both for the entire dataset (overall fairness) and separately for specific demographic groups (e.g., gender) by isolating subgroup data and comparing prediction outcomes.
# The code quantifies disparities between groups to assess whether the model exhibits biases, providing insights into the trade-offs between fairness and accuracy.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

# 1. Load and preprocess data (same as before)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain',
    'capital-loss', 'hours-per-week', 'native-country', 'income'
]
df = pd.read_csv(url, header=None, names=column_names, na_values=' ?', skipinitialspace=True)
df.dropna(inplace=True)

# Convert categorical variables to dummy variables
df = pd.get_dummies(df, columns=['workclass', 'education', 'marital-status',
                                 'occupation', 'relationship', 'race',
                                 'sex', 'native-country'], drop_first=True)

# Convert income to binary (0: <=50K, 1: >50K)
df['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

# 2. Separate features and target
X = df.drop('income', axis=1)
y = df['income']

# 3. Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

# 4. Train a simple model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 5. Make predictions
y_pred = model.predict(X_test)

# 6. Evaluate fairness metrics for the entire dataset (overall)
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
statistical_parity_overall = (TP + FP) / len(y_test)  # Proportion predicted positive
true_positive_rate_overall = TP / (TP + FN)

print(f"Overall Statistical Parity: {statistical_parity_overall:.2f}")
print(f"Overall True Positive Rate (TPR): {true_positive_rate_overall:.2f}\n")

# 7. Evaluate metrics by a protected attribute, e.g., 'sex_Male'
#    (assuming the one-hot encoded column name is 'sex_Male').
protected_attr = 'sex_Male'

# Create masks for subgroups
group_0_mask = (X_test[protected_attr] == 0)  # e.g., female
group_1_mask = (X_test[protected_attr] == 1)  # e.g., male

# Subgroup predictions and true labels
y_pred_group_0 = y_pred[group_0_mask]
y_true_group_0 = y_test[group_0_mask]
y_pred_group_1 = y_pred[group_1_mask]
y_true_group_1 = y_test[group_1_mask]

# Confusion matrices for each subgroup
tn0, fp0, fn0, tp0 = confusion_matrix(y_true_group_0, y_pred_group_0).ravel()
tn1, fp1, fn1, tp1 = confusion_matrix(y_true_group_1, y_pred_group_1).ravel()

# Calculate Statistical Parity for each group
sp_group_0 = (tp0 + fp0) / len(y_true_group_0)
sp_group_1 = (tp1 + fp1) / len(y_true_group_1)

# Calculate True Positive Rate for each group
tpr_group_0 = tp0 / (tp0 + fn0)
tpr_group_1 = tp1 / (tp1 + fn1)

print("=== Metrics by 'sex' Subgroup ===")
print(f"Group 0 (Female) - Statistical Parity: {sp_group_0:.2f}, TPR: {tpr_group_0:.2f}")
print(f"Group 1 (Male)   - Statistical Parity: {sp_group_1:.2f}, TPR: {tpr_group_1:.2f}")

# Calculate differences
print(f"\nStatistical Parity Difference (Group 1 - Group 0): {sp_group_1 - sp_group_0:.2f}")
print(f"TPR Difference (Group 1 - Group 0): {tpr_group_1 - tpr_group_0:.2f}")


### **3.2. Balancing Act: Navigating Metric Trade-offs**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_03_02.png" width="250">



>* Fairness metrics evaluate equitable AI outcomes.
* Trade-offs exist between fairness and efficiency.

>* Equal representation vs. merit-based selection conflict.
* Align metrics with values and expectations.

>* Adapt fairness metrics to evolving societal norms.
* Engage stakeholders for ethical metric alignment.



### **3.3. Challenges in Implementing Fairness in Practice**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_03_03.png" width="250">



>* Bias in historical data affects AI fairness.
* Fairness varies across contexts and requires monitoring.

>* Fairness vs. accuracy trade-offs in AI systems.
* Stakeholder disagreements on fairness metric priorities.

>* Technical barriers and lack of expertise.
* Need for governance and ethical culture.



In [ ]:
#@title Python Code - Challenges in Implementing Fairness in Practice

# This code demonstrates how to train a convolutional neural network on the CIFAR-10 dataset while introducing a synthetic protected attribute to illustrate fairness challenges.
# It uses weighted sampling to address data imbalances between protected groups and applies a fairness-aware loss function to penalize performance disparities.
# The script measures accuracy across groups and calculates a fairness gap to showcase how well the model is balancing group performance.

# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision
import torchvision.transforms as T
import numpy as np
import random

# Define a custom dataset class that wraps CIFAR-10 and adds a synthetic protected attribute
class CIFAR10WithProtectedAttribute(Dataset):
    """
    Extends the standard CIFAR-10 dataset to include a synthetic
    protected attribute. For demonstration, we randomly assign
    protected_attribute = 0 or 1 based on some distribution skewed
    by the label.
    """
    # This initializer sets up the CIFAR-10 dataset and creates protected attributes
    def __init__(self, root, train=True, transform=None, download=True):
        # Create an instance of the CIFAR-10 dataset
        self.cifar10 = torchvision.datasets.CIFAR10(
            root=root,
            train=train,
            transform=transform,
            download=download
        )
        # Initialize an empty list to store synthetic protected attributes
        self.protected_attributes = []
        # Loop over the dataset to assign a protected attribute to each sample
        for idx in range(len(self.cifar10)):
            # Retrieve the label for the current sample
            label = self.cifar10[idx][1]
            # If the label is less than 5, bias the probability toward protected_attr=0
            if label < 5:
                p = 0.8
            # Otherwise, bias toward protected_attr=1
            else:
                p = 0.2
            # Generate the protected attribute based on the random probability
            protected_attr = 0 if random.random() < p else 1
            # Append the protected attribute to the list
            self.protected_attributes.append(protected_attr)
    # This method allows using the len() function on our dataset
    def __len__(self):
        return len(self.cifar10)
    # This method retrieves an item (image, label, protected attribute) by index
    def __getitem__(self, idx):
        # Get the image and label from the underlying CIFAR-10
        img, label = self.cifar10[idx]
        # Retrieve the corresponding protected attribute
        protected_attr = self.protected_attributes[idx]
        # Return all three for later use in training
        return img, label, protected_attr

# Create a set of transformations for training images, including data augmentation
transform_train = T.Compose([
    # Randomly flip images horizontally
    T.RandomHorizontalFlip(),
    # Randomly crop images with a padding of 4 pixels
    T.RandomCrop(32, padding=4),
    # Convert images to PyTorch tensors
    T.ToTensor(),
    # Normalize using the mean and standard deviation of CIFAR-10
    T.Normalize((0.4914, 0.4822, 0.4465),
                (0.2470, 0.2435, 0.2616))
])

# Create a simpler set of transformations for test images
transform_test = T.Compose([
    # Convert images to PyTorch tensors
    T.ToTensor(),
    # Normalize using the mean and standard deviation of CIFAR-10
    T.Normalize((0.4914, 0.4822, 0.4465),
                (0.2470, 0.2435, 0.2616))
])

# Create the training dataset with the custom dataset class
train_dataset = CIFAR10WithProtectedAttribute(
    root='./data',
    train=True,
    transform=transform_train,
    download=True
)

# Create the test dataset
test_dataset = CIFAR10WithProtectedAttribute(
    root='./data',
    train=False,
    transform=transform_test,
    download=True
)

# Import Counter from collections to count occurrences of (label, protected_attr) pairs
from collections import Counter

# Initialize a Counter to store frequencies of each (label, protected_attr)
train_counts = Counter()
# Loop through the entire training dataset to compute the frequencies
for i in range(len(train_dataset)):
    _, label, prot = train_dataset[i]
    train_counts[(label, prot)] += 1

# Find the maximum frequency among all pairs
max_count = max(train_counts.values())
# Initialize a list for sample weights
weights = []
# Loop over each sample to assign a weight based on its pair frequency
for i in range(len(train_dataset)):
    _, label, prot = train_dataset[i]
    # Invert the frequency so that underrepresented samples get higher weights
    weights.append(max_count / train_counts[(label, prot)])

# Create a WeightedRandomSampler to sample data in a balanced manner across groups
train_sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(train_dataset),
    replacement=True
)

# Define the training DataLoader, which uses the weighted sampler
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    sampler=train_sampler,
    num_workers=2
)

# Define the test DataLoader (no special sampler needed)
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

# Create a simple CNN model class
class SimpleCNN(nn.Module):
    # The constructor sets up the network layers
    def __init__(self, num_classes=10):
        # Call the parent constructor
        super(SimpleCNN, self).__init__()
        # Define a feature extraction sequence of convolution, batch norm, activation, pooling
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4))
        )
        # Define a classification head (flatten + fully connected layers)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    # Define the forward pass of the CNN
    def forward(self, x):
        # Pass the input through the features
        x = self.features(x)
        # Pass the extracted features through the classifier
        x = self.classifier(x)
        # Return the output logits
        return x

# Instantiate the CNN model with 10 output classes (for CIFAR-10)
model = SimpleCNN(num_classes=10)

# Detect if a GPU is available and move the model to GPU if possible
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Define a cross-entropy loss with 'none' reduction so we can compute group-wise losses
criterion_ce = nn.CrossEntropyLoss(reduction='none')

# Define a custom fairness-aware loss function
def fairness_aware_loss(outputs, targets, prot_attrs):
    """
    outputs: (batch_size, num_classes)
    targets: (batch_size,) ground-truth labels
    prot_attrs: (batch_size,) protected attributes (0 or 1 in this example)
    Returns scalar loss = cross-entropy + alpha * fairness_penalty
    """
    # Compute the cross-entropy loss for each example individually
    ce_loss_all = criterion_ce(outputs, targets)
    # Create boolean masks for each protected group
    mask_group0 = (prot_attrs == 0)
    mask_group1 = (prot_attrs == 1)
    # Compute the average loss for group0
    if mask_group0.any():
        loss_group0 = ce_loss_all[mask_group0].mean()
    else:
        loss_group0 = 0.0
    # Compute the average loss for group1
    if mask_group1.any():
        loss_group1 = ce_loss_all[mask_group1].mean()
    else:
        loss_group1 = 0.0
    # Compute a penalty based on the absolute difference in average loss between the groups
    fairness_penalty = torch.abs(loss_group0 - loss_group1)
    # Weight this penalty by a hyperparameter alpha
    alpha = 0.2
    # Combine the mean cross-entropy loss with the fairness penalty
    loss = ce_loss_all.mean() + alpha * fairness_penalty
    # Return the final loss scalar
    return loss

# Define an optimizer (Adam) to train the CNN
optimizer = optim.Adam(model.parameters(), lr=0.001)
# Set the number of epochs for training
num_epochs = 5

# Define an evaluation function to measure both accuracy and fairness metrics
def evaluate(model, data_loader):
    # Set the model to evaluation mode
    model.eval()
    # Track total correct predictions and samples
    correct = 0
    total = 0
    # Prepare lists to collect group-wise predictions/labels
    group0_labels = []
    group0_preds = []
    group1_labels = []
    group1_preds = []
    # Disable gradient computation for efficiency during evaluation
    with torch.no_grad():
        # Iterate through the data loader
        for images, labels, prot_attrs in data_loader:
            # Move images, labels, and protected attributes to the GPU if available
            images, labels, prot_attrs = images.to(device), labels.to(device), prot_attrs.to(device)
            # Forward pass through the model
            outputs = model(images)
            # Get the predicted class by selecting the argmax
            _, predicted = torch.max(outputs, 1)
            # Update the total count
            total += labels.size(0)
            # Count how many predictions are correct
            correct += (predicted == labels).sum().item()
            # Separate predictions by protected attribute group
            mask0 = (prot_attrs == 0)
            mask1 = (prot_attrs == 1)
            group0_labels += labels[mask0].tolist()
            group0_preds += predicted[mask0].tolist()
            group1_labels += labels[mask1].tolist()
            group1_preds += predicted[mask1].tolist()
    # Calculate the overall accuracy
    accuracy = 100.0 * correct / total
    # Calculate group-wise accuracy
    group0_acc = 0.0
    group1_acc = 0.0
    # Compute accuracy for group0 if any samples exist
    if len(group0_labels) > 0:
        group0_correct = sum([
            1 for i in range(len(group0_labels))
            if group0_labels[i] == group0_preds[i]
        ])
        group0_acc = 100.0 * group0_correct / len(group0_labels)
    # Compute accuracy for group1 if any samples exist
    if len(group1_labels) > 0:
        group1_correct = sum([
            1 for i in range(len(group1_labels))
            if group1_labels[i] == group1_preds[i]
        ])
        group1_acc = 100.0 * group1_correct / len(group1_labels)
    # Define a fairness gap as the absolute difference between group accuracies
    fairness_gap = abs(group0_acc - group1_acc)
    # Return the overall accuracy, group accuracies, and the fairness gap
    return accuracy, group0_acc, group1_acc, fairness_gap

# Train the model for the specified number of epochs
for epoch in range(num_epochs):
    # Set the model to training mode
    model.train()
    # Initialize a variable to keep track of running loss
    running_loss = 0.0
    # Iterate through the weighted training loader
    for i, (images, labels, prot_attrs) in enumerate(train_loader):
        # Move images, labels, and protected attributes to the GPU if available
        images, labels, prot_attrs = images.to(device), labels.to(device), prot_attrs.to(device)
        # Zero the gradients in the optimizer
        optimizer.zero_grad()
        # Forward pass through the model
        outputs = model(images)
        # Compute our custom fairness-aware loss
        loss = fairness_aware_loss(outputs, labels, prot_attrs)
        # Backpropagation
        loss.backward()
        # Update the model parameters
        optimizer.step()
        # Accumulate loss for display
        running_loss += loss.item()
    # Evaluate on the training data to see how we are doing
    train_acc, g0_acc, g1_acc, fairness_gap = evaluate(model, train_loader)
    # Evaluate on the test data to see generalization performance
    test_acc, g0_acc_test, g1_acc_test, fairness_gap_test = evaluate(model, test_loader)
    # Print epoch training stats, including fairness metrics for each group
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f} "
          f"Train Acc: {train_acc:.2f}% "
          f"Test Acc: {test_acc:.2f}%")
    print(f"  -> Group0 Acc (Train): {g0_acc:.2f}%, Group1 Acc (Train): {g1_acc:.2f}%, Gap: {fairness_gap:.2f}%")
    print(f"  -> Group0 Acc (Test): {g0_acc_test:.2f}%, Group1 Acc (Test): {g1_acc_test:.2f}%, Gap: {fairness_gap_test:.2f}%")
    print("-"*80)

# Print final summary after training completes
print("Training complete. Below are the final observed metrics on the test set:")
final_test_acc, final_g0_acc, final_g1_acc, final_fairness_gap = evaluate(model, test_loader)
print(f"Final Test Accuracy: {final_test_acc:.2f}%")
print(f"Final Group0 Accuracy: {final_g0_acc:.2f}%")
print(f"Final Group1 Accuracy: {final_g1_acc:.2f}%")
print(f"Final Fairness Gap (Absolute Difference): {final_fairness_gap:.2f}%")


### **3.4. Impact on Public Perception and Brand Integrity**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_03_04.png" width="250">



>* Fairness metrics enhance reputation and trust.
* Bias leads to backlash and financial loss.

>* Fairness metrics differentiate brands in competitive markets.
* Neglecting fairness risks reputational damage and penalties.

>* Ethical AI impacts brand integrity significantly.
* Fairness metrics enhance public image and trust.



### **3.5. Navigating Ethical Challenges in Fairness Standards**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20A/image_03_05.png" width="250">



>* Fairness metrics may reinforce societal biases.
* Engage communities for socially responsible standards.

>* Conflicting fairness metrics create ethical dilemmas.
* Balancing metrics requires transparency and accountability.

>* Cultural differences affect fairness perceptions globally.
* Multinationals must adapt AI to local contexts.



# <font color="#418FDE" size="6.5" uppercase>**Foundations of Bias and Fairness in AI**</font>


In this lecture, you learned to:
- Identify common sources of bias in AI systems.
- Distinguish between bias and fairness in AI.
- Evaluate various fairness metrics and their trade-offs.

In the next Lecture (Lecture B), we will go over 'Bias Mitigation Techniques and Case Study'